# EE/CS 148B HW3 — Colab Setup

Open this notebook in Colab, switch to a GPU runtime (T4/L4 for §2/§3/§4, A100 for §5/§6), and run the cells in order. The setup cells will clone the repo, install dependencies, and import everything.

## 1. Clone the repository

By default this clones from GitHub. If you'd rather use a copy on Google Drive, set `USE_DRIVE = True` in the next cell and edit `DRIVE_REPO_ROOT` to point to your folder.

In [ ]:
from pathlib import Path

GITHUB_REPO = 'https://github.com/trevorbchen/148hw3.git'
BRANCH = 'main'

USE_DRIVE = False
DRIVE_REPO_ROOT = Path('/content/drive/MyDrive/hw3/')  # edit if you use Drive
LOCAL_REPO_ROOT = Path('/content/hw3')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = DRIVE_REPO_ROOT
    assert REPO_ROOT.exists(), f'Repo root does not exist on Drive: {REPO_ROOT}'
else:
    REPO_ROOT = LOCAL_REPO_ROOT
    if not REPO_ROOT.exists():
        !git clone --branch {BRANCH} {GITHUB_REPO} {REPO_ROOT}
    else:
        print(f'{REPO_ROOT} already exists — pulling latest')
        !cd {REPO_ROOT} && git pull

print('Using repo:', REPO_ROOT)

## 2. Install dependencies

In [ ]:
%%capture
!pip -q install -U torch torchvision transformers datasets "sentence-transformers<4.0" accelerate pillow tqdm matplotlib wandb pyyaml einops pytest pytest-cov
!pip -q install ninja packaging

In [ ]:
# FlashAttention-2 is only required for §5 (VLM training). Skip if you're
# only doing §2/§3/§4 — the install takes 5+ minutes.
INSTALL_FLASH_ATTN = False
if INSTALL_FLASH_ATTN:
    !pip -q install flash-attn --no-build-isolation

> **Important — restart the runtime now.** Colab boots with old versions of `numpy`, `torch`, etc. preloaded; the imports below will fail with a `numpy.dtype size changed` ABI error if you don't restart after the pip installs above.
>
> Click **Runtime → Restart session** (or run the next cell), then **start running cells from this point** — you do NOT need to re-run the pip install cells.

In [ ]:
# Programmatic kernel restart (equivalent to Runtime -> Restart session).
# After the kernel restarts, RESUME from the next cell — do not re-run the pip cells.
import os
os.kill(os.getpid(), 9)

## 3. Set up paths and import HW3

In [ ]:
import os
import sys
from pathlib import Path

# Re-derive REPO_ROOT after the kernel restart above. Edit if you changed it in cell 1.
REPO_ROOT = Path('/content/hw3')
if not REPO_ROOT.exists():
    REPO_ROOT = Path('/content/drive/MyDrive/hw3/')  # fallback for Drive users
assert REPO_ROOT.exists(), f'Repo root not found: {REPO_ROOT}'

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'basics'))
os.chdir(REPO_ROOT)
print('cwd =', os.getcwd())

In [ ]:
import gc
import math
import random
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

import basics
import vlm
from basics.lora import LoRALinear, apply_lora_to_attention
from basics.model import Block, MultiHeadAttention
from basics.rope import RoPE1D, RoPE2D
from basics.text_encoder import FrozenTextEncoder
from basics.vit import PatchEmbeddings, ViT
from vlm.clip import ProjectionHeads, clip_loss, init_logit_scale
from vlm.data import (
    EUROSAT_CLASSES,
    build_clevr_loaders,
    build_eurosat_loaders,
    build_resisc45_loaders,
)
from vlm.eval import batch_clevr_accuracy, clevr_exact_match, zeroshot_classification_accuracy
from vlm.masking import build_image_bidir_mask
from vlm.model import VisionLanguageModel
from vlm.projector import VisionLanguageProjector

SEED = 0
set_seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.set_float32_matmul_precision('high')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', DEVICE)
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    try:
        print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True, check=False).stdout)
    except FileNotFoundError:
        pass

CONFIGS = {
    'clip': REPO_ROOT / 'configs' / 'clip_eurosat.yaml',
    'lora': REPO_ROOT / 'configs' / 'lora_resisc.yaml',
    'vlm': REPO_ROOT / 'configs' / 'vlm_clevr.yaml',
}
for name, path in CONFIGS.items():
    assert path.exists(), f'Missing {name} config: {path}'
    with open(path) as f:
        _ = yaml.safe_load(f)

print('HW3 imports and configs loaded.')

## 4. (Optional) Run the unit tests

Quick sanity check that the implementations in `basics/` and `vlm/clip.py` are working.

In [ ]:
!python -m pytest tests/ -v

## 5. Run an experiment

Each script writes to `runs/<experiment>/`:
- `log.csv` — per-step metrics
- `metrics.json` — final summary
- `figures/*.png` — loss / accuracy / LR plots
- `best.pt` — best checkpoint

On Colab these live under `/content/hw3/runs/`. Download with the file browser or `google.colab.files.download(...)`.

In [ ]:
# §3 — CLIP pretraining on EuroSAT (writes runs/clip_eurosat/)
!python scripts/pretrain_clip.py --config configs/clip_eurosat.yaml

In [ ]:
# §4 — RESISC45 fine-tuning. Run all three for a comparison.
!python scripts/finetune_resisc.py --config configs/lora_resisc.yaml --method linear_probe --pretrained runs/clip_eurosat/best.pt
!python scripts/finetune_resisc.py --config configs/lora_resisc.yaml --method lora --rank 8 --pretrained runs/clip_eurosat/best.pt
!python scripts/finetune_resisc.py --config configs/lora_resisc.yaml --method full_ft --pretrained runs/clip_eurosat/best.pt

In [ ]:
# §5 — VLM training on CLEVR (needs FlashAttention-2; flip INSTALL_FLASH_ATTN above and re-run).
# Run scripts/download_clevr.py first if you haven't.
!python scripts/train_vlm.py --config configs/vlm_clevr.yaml \
    --pretrained-vit runs/clip_eurosat/best.pt \
    --injection all_patches --mask-mode image_bidir --freeze-config A

## 6. View saved figures inline

In [ ]:
from IPython.display import Image, display
from pathlib import Path
import json

for run in sorted(Path('runs').glob('*')):
    if not run.is_dir():
        continue
    print(f"=== {run} ===")
    metrics_path = run / 'metrics.json'
    if metrics_path.exists():
        print(json.dumps(json.loads(metrics_path.read_text()), indent=2))
    for fig in sorted(run.glob('figures/*.png')):
        print(fig)
        display(Image(str(fig)))

## Your HW3 code starts here